# Notebook 03 — System Implications

**Purpose:** Translate the conditional Monte Carlo commitment depths from
notebook 02 into accredited MW, avoided capacity cost, and interconnection
acceleration NPV. Reproduces Nature Energy manuscript §6.

**Inputs** (from notebook 02):
- `outputs/contracts/cascade_parameters.json`
- `outputs/contracts/conditional_mc_results.json`

**Outputs:**
- `outputs/contracts/final_results.json`
- `outputs/tables/system_implications_summary.csv`

---

## Notebook Architecture

| Part | Section | Contents |
|---|---|---|
| **0** | Setup | Imports, contract loading, sanity checks |
| **1** | Accredited Capacity (§6) | ELCC application, reference cases, per-GW curve |
| **2** | Avoided Capacity Cost (§6) | E3 levelized CT, CEJA sensitivity |
| **3** | Interconnection Acceleration NPV (§6) | Foregone revenue, sensitivity grid |
| **4** | Summary and Exports | Consolidated table, validation, final_results.json |

## Part 0: Setup

- **0.1** Imports and configuration
- **0.2** Load contracts from notebook 02
- **0.3** Contract sanity checks

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-1: IMPORTS AND CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path

# ─── REPO_ROOT resolver (same pattern as notebooks 01 and 02) ────────────────
_cwd = Path.cwd()
if (_cwd / 'notebooks').is_dir():
    REPO_ROOT = _cwd
elif _cwd.name == 'notebooks' and (_cwd.parent / 'notebooks').is_dir():
    REPO_ROOT = _cwd.parent
else:
    REPO_ROOT = _cwd

CONTRACTS_DIR = REPO_ROOT / 'outputs' / 'contracts'
TABLES_DIR    = REPO_ROOT / 'outputs' / 'tables'
FIGURES_DIR   = REPO_ROOT / 'outputs' / 'figures'

os.makedirs(CONTRACTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"REPO_ROOT:     {REPO_ROOT}")
print(f"CONTRACTS_DIR: {CONTRACTS_DIR}")

REPO_ROOT:     C:\Users\dunla\repos\data-center-flexibility-resource-adequacy
CONTRACTS_DIR: C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\outputs\contracts


### 0.2 Load Contracts from Notebook 02

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-2: LOAD CONTRACTS FROM NOTEBOOK 02
# ══════════════════════════════════════════════════════════════════════════════

with open(CONTRACTS_DIR / 'cascade_parameters.json') as f:
    cascade_params = json.load(f)

with open(CONTRACTS_DIR / 'conditional_mc_results.json') as f:
    conditional_mc = json.load(f)

print(f"cascade_parameters.json:")
print(f"  Central cascade product:  {cascade_params['cascade_product']['central']:.4f}")
print(f"  Commitment depth baseline: {cascade_params['commitment_depth_baseline']:.1%}")
print(f"  DVFS floor:               {cascade_params['dvfs_floor_facility']:.1%}")
print()
print(f"conditional_mc_results.json:")
print(f"  Single facility (500 MW): mean={conditional_mc['single_facility_500mw']['mean_commitment_depth']:.1%}")
print(f"  Empirical fleet:          mean={conditional_mc['empirical_fleet']['mean_commitment_depth']:.1%}")
print(f"  Per-GW sweep entries:     {len(conditional_mc['per_gw_sweep'])}")
print(f"  Contention onset:         {conditional_mc['contention_onset_gw']} GW")

cascade_parameters.json:
  Central cascade product:  0.0384
  Commitment depth baseline: 20.4%
  DVFS floor:               17.5%

conditional_mc_results.json:
  Single facility (500 MW): mean=46.9%
  Empirical fleet:          mean=28.9%
  Per-GW sweep entries:     11
  Contention onset:         3.0 GW


### 0.3 Contract Sanity Checks

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-3: CONTRACT SANITY CHECKS
# ══════════════════════════════════════════════════════════════════════════════

assert cascade_params['cascade_product']['central'] > 0, "Cascade product is zero or negative"
assert 0.15 < cascade_params['commitment_depth_baseline'] < 0.30, (
    f"Commitment depth baseline {cascade_params['commitment_depth_baseline']} outside expected range"
)
assert len(cascade_params['parameters']) == 10, "Expected 10 cascade parameters"

assert 0.30 < conditional_mc['single_facility_500mw']['mean_commitment_depth'] < 0.70, (
    "Single facility mean depth outside plausible range"
)
assert len(conditional_mc['per_gw_sweep']) >= 5, "Per-GW sweep has too few entries"

# Build sweep DataFrame for downstream use
sweep_df = pd.DataFrame(conditional_mc['per_gw_sweep'])
assert 'fleet_gw' in sweep_df.columns
assert 'mean' in sweep_df.columns

print(f"✓ Contract sanity checks passed")
print(f"  Cascade parameters: {len(cascade_params['parameters'])}")
print(f"  Sweep fleet sizes:  {list(sweep_df['fleet_gw'].values)}")

✓ Contract sanity checks passed
  Cascade parameters: 10
  Sweep fleet sizes:  [np.float64(0.5), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(8.0), np.float64(10.0), np.float64(12.0), np.float64(15.0)]


## Part 1: Accredited Capacity (§6)

- **1.1** ELCC application and reference cases
- **1.2** Per-GW accredited MW curve

Translates conditional MC commitment depths into accredited MW using PJM's
92% ELCC for demand response resources (ref. 29). The commitment depth ×
fleet MW gives curtailable load; the ELCC derate gives accredited capacity
that enters the capacity market.

### 1.1 ELCC Application and Reference Cases

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1-1: ACCREDITED MW FROM CONDITIONAL MC COMMITMENT DEPTH
# ══════════════════════════════════════════════════════════════════════════════
# §6 calculation: committed_mw = depth × fleet_mw
#                 accredited_mw = committed_mw × ELCC
# ══════════════════════════════════════════════════════════════════════════════

DR_ELCC = 0.92  # PJM DR class rating (NE paper §6, ref. 29)

# ─── Per-GW curve ────────────────────────────────────────────────────────────
sweep_df['committed_mw'] = sweep_df['mean'] * sweep_df['fleet_mw']
sweep_df['accredited_mw'] = sweep_df['committed_mw'] * DR_ELCC

# ─── Reference cases (1 GW and 10 GW) ───────────────────────────────────────
REF_CASES = [1.0, 10.0]
ref_rows = []
for _gw in REF_CASES:
    _row = sweep_df[sweep_df['fleet_gw'] == _gw].iloc[0]
    ref_rows.append({
        'fleet_gw': _gw,
        'fleet_mw': _row['fleet_mw'],
        'mean_depth': _row['mean'],
        'committed_mw': _row['committed_mw'],
        'accredited_mw': _row['accredited_mw'],
    })

ref_df = pd.DataFrame(ref_rows)

# ─── Print ───────────────────────────────────────────────────────────────────
print("ACCREDITED CAPACITY BY REFERENCE FLEET SIZE")
print("=" * 70)
print(f"  ELCC (PJM DR class): {DR_ELCC:.0%}")
print()
for _, _r in ref_df.iterrows():
    print(f"  {_r['fleet_gw']:.0f} GW fleet:")
    print(f"    Mean commitment depth: {_r['mean_depth']:.1%}")
    print(f"    Curtailable MW:        {_r['committed_mw']:,.0f}")
    print(f"    Accredited MW:         {_r['accredited_mw']:,.0f}")
    print()

# ─── Full sweep table ────────────────────────────────────────────────────────
print("PER-GW ACCREDITED CAPACITY CURVE")
print("─" * 70)
print(f"  {'Fleet':>6} | {'Depth':>7} | {'Curtailable':>12} | {'Accredited':>11}")
print("  " + "─" * 55)
for _, _r in sweep_df.iterrows():
    print(f"  {_r['fleet_gw']:>5.1f}GW | {_r['mean']:>6.1%} | "
          f"{_r['committed_mw']:>10,.0f} MW | {_r['accredited_mw']:>9,.0f} MW")

ACCREDITED CAPACITY BY REFERENCE FLEET SIZE
  ELCC (PJM DR class): 92%

  1 GW fleet:
    Mean commitment depth: 46.4%
    Curtailable MW:        464
    Accredited MW:         427

  10 GW fleet:
    Mean commitment depth: 26.9%
    Curtailable MW:        2,689
    Accredited MW:         2,474

PER-GW ACCREDITED CAPACITY CURVE
──────────────────────────────────────────────────────────────────────
   Fleet |   Depth |  Curtailable |  Accredited
  ───────────────────────────────────────────────────────
    0.5GW |  46.9% |        234 MW |       216 MW
    1.0GW |  46.4% |        464 MW |       427 MW
    2.0GW |  45.2% |        904 MW |       832 MW
    3.0GW |  42.5% |      1,275 MW |     1,173 MW
    4.0GW |  39.0% |      1,562 MW |     1,437 MW
    5.0GW |  35.8% |      1,789 MW |     1,646 MW
    6.0GW |  33.0% |      1,983 MW |     1,824 MW
    8.0GW |  29.2% |      2,339 MW |     2,152 MW
   10.0GW |  26.9% |      2,689 MW |     2,474 MW
   12.0GW |  25.3% |      3,039 MW |     2,

## Part 2: Avoided Capacity Cost (§6)

- **2.1** E3 levelized CT cost application
- **2.2** CEJA sensitivity

Computes the annualized system cost that accredited spatial migration capacity
would displace, using the E3 2025 Resource Adequacy Study levelized CT cost
as the marginal capacity resource.

### 2.1 Avoided Capacity Cost

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2-1: AVOIDED CAPACITY COST (E3 levelized CT)
# ══════════════════════════════════════════════════════════════════════════════
# §6: accredited_mw × $/MW-yr = annual avoided capacity procurement cost
# ══════════════════════════════════════════════════════════════════════════════

E3_CT_LEVELIZED_COST = 180_000   # $/MW-yr (E3 2025 IL RA Study, Table 6-1)
E3_CT_LEVELIZED_CEJA = 205_000   # $/MW-yr (Illinois CEJA sensitivity)

# ─── Apply to reference cases ────────────────────────────────────────────────
for _idx, _r in ref_df.iterrows():
    ref_df.loc[_idx, 'annual_avoided_base'] = _r['accredited_mw'] * E3_CT_LEVELIZED_COST
    ref_df.loc[_idx, 'annual_avoided_ceja'] = _r['accredited_mw'] * E3_CT_LEVELIZED_CEJA

# ─── Apply to full sweep ────────────────────────────────────────────────────
sweep_df['annual_avoided_base'] = sweep_df['accredited_mw'] * E3_CT_LEVELIZED_COST
sweep_df['annual_avoided_ceja'] = sweep_df['accredited_mw'] * E3_CT_LEVELIZED_CEJA

# ─── Print ───────────────────────────────────────────────────────────────────
print("AVOIDED CAPACITY COST")
print("=" * 70)
print(f"  E3 levelized CT cost:  ${E3_CT_LEVELIZED_COST:,}/MW-yr (base)")
print(f"  E3 CEJA sensitivity:   ${E3_CT_LEVELIZED_CEJA:,}/MW-yr")
print()

for _, _r in ref_df.iterrows():
    print(f"  {_r['fleet_gw']:.0f} GW fleet:")
    print(f"    Accredited MW:              {_r['accredited_mw']:>10,.0f}")
    print(f"    Annual avoided (base):      ${_r['annual_avoided_base']/1e6:>10,.0f}M")
    print(f"    Annual avoided (CEJA):      ${_r['annual_avoided_ceja']/1e6:>10,.0f}M")
    print()

# ─── Diminishing returns note ────────────────────────────────────────────────
_contention_gw = conditional_mc['contention_onset_gw']
print(f"  Contention onset: {_contention_gw} GW (>50% of MC draws destination-constrained)")
print(f"  Beyond {_contention_gw} GW, per-MW avoided cost declines as depth decreases with scale.")

AVOIDED CAPACITY COST
  E3 levelized CT cost:  $180,000/MW-yr (base)
  E3 CEJA sensitivity:   $205,000/MW-yr

  1 GW fleet:
    Accredited MW:                     427
    Annual avoided (base):      $        77M
    Annual avoided (CEJA):      $        87M

  10 GW fleet:
    Accredited MW:                   2,474
    Annual avoided (base):      $       445M
    Annual avoided (CEJA):      $       507M

  Contention onset: 3.0 GW (>50% of MC draws destination-constrained)
  Beyond 3.0 GW, per-MW avoided cost declines as depth decreases with scale.


## Part 3: Interconnection Acceleration NPV (§6)

- **3.1** Assumptions and NPV computation
- **3.2** Dominance analysis and policy implication

The primary economic incentive for spatial migration investment is not
capacity market revenue but interconnection queue acceleration. If a
flexible interconnection agreement (FIA) allows a facility to energize
before network upgrades complete, the present value of avoided delay
depends on foregone compute revenue during the waiting period.

### 3.1 IX Queue NPV Computation

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3-1: IX QUEUE NPV — BEHAVIORAL INCENTIVE
# ══════════════════════════════════════════════════════════════════════════════
# Ported from Bartlett v18 Cell 5-1. The IX acceleration NPV dwarfs capacity
# market revenue by orders of magnitude — capacity is the compliance hook,
# IX speed is the payoff.
#
# Infrastructure constants below are from v18 Cell 0-2 (frozen at migration).
# Commitment depth and DVFS floor are read from notebook 02 contracts.
# ══════════════════════════════════════════════════════════════════════════════

# ─── Infrastructure constants (from v18 Cell 0-2) ───────────────────────────
SYSTEM_MAX_POWER_KW = 10.2        # DGX H100 node max power
GPUS_PER_SYSTEM     = 8
FACILITY_PUE        = 1.30        # Uptime Institute 2024
GPU_IT_POWER_KW     = SYSTEM_MAX_POWER_KW / GPUS_PER_SYSTEM          # 1.275 kW/GPU
GPU_GRID_POWER_KW   = GPU_IT_POWER_KW * FACILITY_PUE                 # 1.6575 kW/GPU
GPU_PER_MW_GRID     = int(1000 / GPU_GRID_POWER_KW)                  # 603 GPUs/MW

GPU_RATE_HR          = 2.20       # Jan 2026 H100 spot rate $/hr
BRA_2027_28_PRICE    = 333.44     # $/MW-day (PJM 2027/28 BRA)
WACC                 = 0.10       # Brattle VRR ATWACC ≈ 9.5%, rounded to 10%

# ─── From contracts (no hardcoding) ──────────────────────────────────────────
FLEX_FRAC               = cascade_params['flex_frac']
DVFS_COMMITMENT_FRAC    = cascade_params['dvfs_floor_facility']
# 1 GW commitment depth from conditional MC per-GW sweep
OPTIMAL_COMMITMENT_FRAC = ref_df.loc[ref_df['fleet_gw'] == 1.0, 'mean_depth'].iloc[0]

print("IX QUEUE NPV — BEHAVIORAL INCENTIVE QUANTIFICATION")
print("=" * 70)

# ─── 1. Reference facility ──────────────────────────────────────────────────
FACILITY_GW    = 1.0
FACILITY_MW    = FACILITY_GW * 1000
UTILIZATION    = 0.80
HOURS_PER_YEAR = 8760

annual_compute_rev = GPU_PER_MW_GRID * GPU_RATE_HR * HOURS_PER_YEAR * UTILIZATION * FACILITY_MW

print(f"Reference facility: {FACILITY_GW:.0f} GW grid-connected")
print(f"GPU density: {GPU_PER_MW_GRID:,} GPUs/MW (grid-metered, PUE={FACILITY_PUE})")
print(f"H100 spot rate: ${GPU_RATE_HR}/hr")
print(f"Utilization: {UTILIZATION:.0%}")
print(f"Annual gross compute revenue: ${annual_compute_rev/1e9:.2f}B/yr")
print()

# ─── 2. IX queue baseline ───────────────────────────────────────────────────
PJM_QUEUE_MEDIAN_YRS = 70 / 12   # LBNL-2024: 5.83 years
PJM_QUEUE_P75_YRS   = 84 / 12   # 7 years

print(f"PJM baseline queue [LBNL-2024]:")
print(f"  Median IR→COD: {PJM_QUEUE_MEDIAN_YRS:.1f} years")
print(f"  P75 IR→COD:    {PJM_QUEUE_P75_YRS:.1f} years")
print()

# ─── 3. FIA acceleration scenarios ──────────────────────────────────────────
acceleration_scenarios = {
    'Conservative': 2.0,
    'Central':      3.0,
    'Optimistic':   4.0,
}

print(f"WACC: {WACC:.0%} (hyperscaler range 8-12%)")
print()

# NPV = annual_rev × annuity_factor(N, r)
print("─" * 70)
print(f"NPV OF IX QUEUE ACCELERATION ({FACILITY_GW:.0f} GW facility)")
print("─" * 70)
print(f"{'Scenario':>14} | {'Accel (yrs)':>11} | {'Annuity Factor':>14} | "
      f"{'NPV (gross)':>12} | {'vs Ann Cap Rev':>14}")
print("-" * 70)

# Annual capacity revenue at 1 GW (DVFS-only baseline for comparison)
flex_mw_1gw = FACILITY_MW * FLEX_FRAC
cap_rev_t3_annual = flex_mw_1gw * BRA_2027_28_PRICE * 365 * DR_ELCC

ix_npv_results = {}
for label, N in acceleration_scenarios.items():
    annuity = (1 - (1 + WACC)**(-N)) / WACC
    npv_gross = annual_compute_rev * annuity
    vs_cap_rev = npv_gross / cap_rev_t3_annual
    ix_npv_results[label] = {'N': N, 'annuity': annuity, 'npv': npv_gross}
    print(f"{label:>14} | {N:>11.0f} | {annuity:>14.3f} | "
          f"${npv_gross/1e9:>10.1f}B | {vs_cap_rev:>12.0f}×")

print()

# ─── 4. Dominance analysis ──────────────────────────────────────────────────
npv_central = ix_npv_results['Central']['npv']

# Revenue differential: DVFS+Spatial vs DVFS-only
cap_rev_diff_annual = (FACILITY_MW * (OPTIMAL_COMMITMENT_FRAC - DVFS_COMMITMENT_FRAC)
                       * BRA_2027_28_PRICE * 365 * DR_ELCC)
cap_diff_npv = cap_rev_diff_annual * ix_npv_results['Central']['annuity']

print("─" * 70)
print("DOMINANCE ANALYSIS")
print("─" * 70)
print(f"Central scenario (3yr acceleration, {FACILITY_GW:.0f} GW):")
print(f"  Commitment depth (1 GW, from conditional MC): {OPTIMAL_COMMITMENT_FRAC:.1%}")
print(f"  DVFS-only floor:                              {DVFS_COMMITMENT_FRAC:.1%}")
print(f"  Gross IX NPV:               ${npv_central/1e9:.1f}B")
print(f"  DVFS+Spatial vs DVFS-only cap differential: ${cap_rev_diff_annual/1e6:.1f}M/yr")
print(f"  Cap differential NPV (3yr):  ${cap_diff_npv/1e6:.0f}M")
print()

breakeven_margin = cap_diff_npv / npv_central
print(f"  IX NPV exceeds cap differential at EBITDA margin > {breakeven_margin:.2%}")
print(f"  Typical hyperscaler cloud EBITDA: 35-45%")
print(f"  → IX incentive dominates at any plausible margin")
print()

# ─── 5. Policy implication ───────────────────────────────────────────────────
cap_rev_spatial_annual = FACILITY_MW * OPTIMAL_COMMITMENT_FRAC * BRA_2027_28_PRICE * 365 * DR_ELCC
cap_rev_dvfs_annual = FACILITY_MW * DVFS_COMMITMENT_FRAC * BRA_2027_28_PRICE * 365 * DR_ELCC

print("─" * 70)
print("MECHANISM DESIGN IMPLICATION")
print("─" * 70)
print(f"  Capacity revenue differential (DVFS+Spatial vs DVFS-only):")
print(f"    ${cap_rev_spatial_annual/1e6:.1f}M/yr ({OPTIMAL_COMMITMENT_FRAC:.1%} depth) vs "
      f"${cap_rev_dvfs_annual/1e6:.1f}M/yr ({DVFS_COMMITMENT_FRAC:.1%} depth)")
print(f"    = ${cap_rev_diff_annual/1e6:.1f}M/yr incremental — the STATED instrument")
print()
print(f"  IX queue NPV (central, gross): ${npv_central/1e9:.1f}B — the ACTUAL incentive")
print(f"  Ratio: {npv_central/cap_rev_diff_annual:.0f}× annual differential")

# ─── Store results ───────────────────────────────────────────────────────────
IX_NPV_CENTRAL     = ix_npv_results['Central']['npv']
IX_NPV_LOW         = ix_npv_results['Conservative']['npv']
IX_NPV_HIGH        = ix_npv_results['Optimistic']['npv']
IX_DOMINANCE_RATIO = npv_central / cap_rev_diff_annual

IX QUEUE NPV — BEHAVIORAL INCENTIVE QUANTIFICATION
Reference facility: 1 GW grid-connected
GPU density: 603 GPUs/MW (grid-metered, PUE=1.3)
H100 spot rate: $2.2/hr
Utilization: 80%
Annual gross compute revenue: $9.30B/yr

PJM baseline queue [LBNL-2024]:
  Median IR→COD: 5.8 years
  P75 IR→COD:    7.0 years

WACC: 10% (hyperscaler range 8-12%)

──────────────────────────────────────────────────────────────────────
NPV OF IX QUEUE ACCELERATION (1 GW facility)
──────────────────────────────────────────────────────────────────────
      Scenario | Accel (yrs) | Annuity Factor |  NPV (gross) | vs Ann Cap Rev
----------------------------------------------------------------------
  Conservative |           2 |          1.736 | $      16.1B |          576×
       Central |           3 |          2.487 | $      23.1B |          826×
    Optimistic |           4 |          3.170 | $      29.5B |         1053×

──────────────────────────────────────────────────────────────────────
DOMINANCE ANALY

## Part 4: Summary and Exports

- **4.1** Consolidated results table
- **4.2** Export final_results.json

### 4.1 Consolidated Results

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4-1: CONSOLIDATED RESULTS TABLE
# ══════════════════════════════════════════════════════════════════════════════
# All §6 numbers in one place for paper update reference.
# ══════════════════════════════════════════════════════════════════════════════

print("CONSOLIDATED §6 RESULTS")
print("=" * 70)
print()
print("ACCREDITED CAPACITY AND AVOIDED COST")
print("─" * 70)
for _, _r in ref_df.iterrows():
    print(f"  {_r['fleet_gw']:.0f} GW fleet:")
    print(f"    Commitment depth (mean):  {_r['mean_depth']:.1%}")
    print(f"    Curtailable MW:           {_r['committed_mw']:,.0f}")
    print(f"    Accredited MW (×{DR_ELCC:.0%}):   {_r['accredited_mw']:,.0f}")
    print(f"    Avoided cost (base):      ${_r['annual_avoided_base']/1e6:,.0f}M/yr")
    print(f"    Avoided cost (CEJA):      ${_r['annual_avoided_ceja']/1e6:,.0f}M/yr")
    print()

print("IX ACCELERATION NPV (1 GW facility)")
print("─" * 70)
print(f"  Conservative (2yr): ${IX_NPV_LOW/1e9:.1f}B")
print(f"  Central (3yr):      ${IX_NPV_CENTRAL/1e9:.1f}B")
print(f"  Optimistic (4yr):   ${IX_NPV_HIGH/1e9:.1f}B")
print(f"  IX / cap differential: {IX_DOMINANCE_RATIO:.0f}×")
print()

print("CONTENTION AND SCALE DEPENDENCE")
print("─" * 70)
print(f"  Contention onset: {conditional_mc['contention_onset_gw']} GW")
print(f"  DVFS floor:       {cascade_params['dvfs_floor_facility']:.1%}")
print(f"  Cascade baseline: {cascade_params['commitment_depth_baseline']:.1%}")

# ─── Export summary CSV ──────────────────────────────────────────────────────
_summary_path = TABLES_DIR / 'system_implications_summary.csv'
sweep_df.to_csv(_summary_path, index=False)
print(f"\n  Wrote {_summary_path.name}")

CONSOLIDATED §6 RESULTS

ACCREDITED CAPACITY AND AVOIDED COST
──────────────────────────────────────────────────────────────────────
  1 GW fleet:
    Commitment depth (mean):  46.4%
    Curtailable MW:           464
    Accredited MW (×92%):   427
    Avoided cost (base):      $77M/yr
    Avoided cost (CEJA):      $87M/yr

  10 GW fleet:
    Commitment depth (mean):  26.9%
    Curtailable MW:           2,689
    Accredited MW (×92%):   2,474
    Avoided cost (base):      $445M/yr
    Avoided cost (CEJA):      $507M/yr

IX ACCELERATION NPV (1 GW facility)
──────────────────────────────────────────────────────────────────────
  Conservative (2yr): $16.1B
  Central (3yr):      $23.1B
  Optimistic (4yr):   $29.5B
  IX / cap differential: 715×

CONTENTION AND SCALE DEPENDENCE
──────────────────────────────────────────────────────────────────────
  Contention onset: 3.0 GW
  DVFS floor:       17.5%
  Cascade baseline: 20.4%

  Wrote system_implications_summary.csv


### 4.2 Export Final Results

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4-2: EXPORT final_results.json
# ══════════════════════════════════════════════════════════════════════════════

final_results = {
    "version": "1.0",
    "produced_by": "notebook 03 — system_implications",
    "elcc": DR_ELCC,
    "e3_ct_levelized_cost": E3_CT_LEVELIZED_COST,
    "e3_ct_levelized_ceja": E3_CT_LEVELIZED_CEJA,
    "reference_cases": ref_df.to_dict(orient='records'),
    "per_gw_curve": sweep_df[['fleet_gw', 'fleet_mw', 'mean', 'committed_mw',
                               'accredited_mw', 'annual_avoided_base']].to_dict(orient='records'),
    "ix_acceleration": {
        "facility_gw": FACILITY_GW,
        "gpu_per_mw_grid": GPU_PER_MW_GRID,
        "gpu_rate_hr": GPU_RATE_HR,
        "annual_compute_rev": annual_compute_rev,
        "wacc": WACC,
        "npv_conservative_2yr": IX_NPV_LOW,
        "npv_central_3yr": IX_NPV_CENTRAL,
        "npv_optimistic_4yr": IX_NPV_HIGH,
        "dominance_ratio": IX_DOMINANCE_RATIO,
    },
    "contention_onset_gw": conditional_mc['contention_onset_gw'],
}

_path = CONTRACTS_DIR / 'final_results.json'
with open(_path, 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print(f"✓ Wrote {_path.name}")
print(f"  Reference cases: {[r['fleet_gw'] for r in final_results['reference_cases']]}")
print(f"  IX NPV central: ${IX_NPV_CENTRAL/1e9:.1f}B")

✓ Wrote final_results.json
  Reference cases: [1.0, 10.0]
  IX NPV central: $23.1B
